In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from gwpy.timeseries import TimeSeries

sys.path.append("../src")
from senal import asd_interpolada, blanquear, convolucionar, espectrograma, filtro_blanqueador, ganancia_pasa_banda, pasa_banda

In [ ]:
EVENTO = "GW150914"
DATOS = Path("../datos")
DURACION = 32
DURACION_FFT = 4
BANDA = (35, 350)
CORTE = (30, 400)
G = 6.674e-11
C = 299792458
MASA_SOL = 1.98847e30
MPC = 3.0857e22
MASA_CHIRP_CATALOGO = 30.2
N_SEGMENTO = 256
PASO = 8
N_FFT = 512
MASAS = np.arange(15, 50.01, 0.25)
DESPLAZAMIENTOS = np.arange(-25, 31)
F_FINAL = 250
FRECUENCIAS_FINALES = [150, 200, 250]
POSICIONES = [-13, -11, -9, -7, -5, -3, 3, 5, 7, 9, 11, 13]
MASAS_INYECTADAS = [20, 30, 40]
DISTANCIA_INYECCION = 870
F_INYECCION = 300

In [ ]:
def masa_en_segundos(masa_chirp):
    return G * masa_chirp * MASA_SOL / C**3


def frecuencia_chirp(tau, masa_chirp):
    return (5 / (256 * tau)) ** (3 / 8) / (np.pi * masa_en_segundos(masa_chirp) ** (5 / 8))


def chirp(t, tc, masa_chirp, distancia, f_final):
    m = masa_en_segundos(masa_chirp)
    h = np.zeros(len(t))
    tau = tc - t
    antes = tau > 0
    f = frecuencia_chirp(tau[antes], masa_chirp)
    fase = -2 * (tau[antes] / (5 * m)) ** (5 / 8)
    amplitud = 4 * C * m ** (5 / 3) * (np.pi * f) ** (2 / 3) / (distancia * MPC)
    bordes = ganancia_pasa_banda(f, (30, f_final), (25, 1.25 * f_final))
    h[antes] = amplitud * bordes * np.cos(fase)
    return h

In [ ]:
tau = np.linspace(1, 0.002, 5000)
f = frecuencia_chirp(tau, 30)
derivada = np.gradient(f, -tau)
teorica = 96 / 5 * np.pi ** (8 / 3) * masa_en_segundos(30) ** (5 / 3) * f ** (11 / 3)
print(f"df/dt numérica frente a la fórmula: diferencia relativa máxima {np.abs(derivada[5:-5] / teorica[5:-5] - 1).max():.1e}")
print(f"De 35 Hz a la fusión: {5 / 256 * (np.pi * 35) ** (-8 / 3) * masa_en_segundos(30) ** (-5 / 3):.3f} s")
print(f"h a 150 Hz y 410 Mpc: {4 * C * masa_en_segundos(30) ** (5 / 3) * (np.pi * 150) ** (2 / 3) / (410 * MPC):.1e}")

In [ ]:
for masa_chirp in [10, 30, 60]:
    plt.plot(-tau, frecuencia_chirp(tau, masa_chirp), label=rf"{masa_chirp} M$_\odot$")
plt.xlim(-1, 0)
plt.ylim(0, 400)
plt.xlabel("Tiempo hasta la fusión (s)")
plt.ylabel("Frecuencia (Hz)")
plt.legend();

In [ ]:
t = np.arange(-0.3, 0, 1 / 4096)
plt.plot(t, chirp(t, 0, 30, 410, F_INYECCION))
plt.xlabel("Tiempo hasta la fusión (s)")
plt.ylabel("h");

In [ ]:
t0 = float((DATOS / f"{EVENTO}_gps.txt").read_text())
datos_h1 = TimeSeries.read(DATOS / f"H1_{EVENTO}_{DURACION}s.hdf5")
datos_l1 = TimeSeries.read(DATOS / f"L1_{EVENTO}_{DURACION}s.hdf5")
fs = int(datos_h1.sample_rate.value)
tiempos = datos_h1.times.value - t0
crudos = {"H1": datos_h1.value, "L1": datos_l1.value}

In [ ]:
def preparar(crudo):
    return pasa_banda(blanquear(crudo, fs, DURACION_FFT), fs, BANDA, CORTE)


def ventana(centro):
    inicio = np.searchsorted(tiempos, centro - 0.8)
    return inicio, tiempos[inicio] - centro


def banco_de_plantillas(crudo, f_final):
    filtro = filtro_blanqueador(asd_interpolada(crudo, fs, DURACION_FFT), 2 * fs)
    _, desfase = ventana(0)
    t = desfase + np.arange(-4096, 4096) / fs
    banco = []
    for masa_chirp in MASAS:
        blanqueada = convolucionar(chirp(t, 0, masa_chirp, 1, f_final), filtro) * np.sqrt(2 / fs)
        filtrada = pasa_banda(blanqueada, fs, BANDA, CORTE)[-4096:]
        _, frecuencias, potencia = espectrograma(filtrada, fs, N_SEGMENTO, PASO, N_FFT)
        banco.append(potencia[(frecuencias >= 30) & (frecuencias <= f_final)])
    return banco


def desplazar(espectro, k):
    movido = np.zeros_like(espectro)
    if k >= 0:
        movido[:, k:] = espectro[:, : espectro.shape[1] - k]
    else:
        movido[:, :k] = espectro[:, -k:]
    return movido


def estimar(filtrado, banco, centro, excluir, f_final):
    tramo = np.abs(tiempos) < 14
    t_columnas, frecuencias, potencia_ruido = espectrograma(filtrado[tramo], fs, N_SEGMENTO, 64, N_FFT)
    t_columnas += tiempos[tramo][0]
    lejos = np.all([np.abs(t_columnas - e) > 1 for e in excluir], axis=0)
    filas = (frecuencias >= 30) & (frecuencias <= f_final)
    nivel = potencia_ruido[filas][:, lejos].mean(axis=1)[:, None]
    inicio, _ = ventana(centro)
    _, _, potencia = espectrograma(filtrado[inicio : inicio + 4096], fs, N_SEGMENTO, PASO, N_FFT)
    exceso = potencia[filas] / nivel - 1
    exceso -= exceso.mean(axis=1, keepdims=True)
    puntaje = np.zeros((len(MASAS), len(DESPLAZAMIENTOS)))
    for i, plantilla in enumerate(banco):
        modelo = plantilla / nivel
        for j, k in enumerate(DESPLAZAMIENTOS):
            movido = desplazar(modelo, k)
            movido -= movido.mean(axis=1, keepdims=True)
            puntaje[i, j] = (exceso * movido).sum() / np.sqrt((movido**2).sum())
    i, j = np.unravel_index(puntaje.argmax(), puntaje.shape)
    return MASAS[i], centro + DESPLAZAMIENTOS[j] * PASO / fs, puntaje

In [ ]:
bancos = {(ifo, f_final): banco_de_plantillas(crudos[ifo], f_final) for ifo in ["H1", "L1"] for f_final in FRECUENCIAS_FINALES}

In [ ]:
errores = {}
for ifo in ["H1", "L1"]:
    for masa_inyectada in MASAS_INYECTADAS:
        distancia = DISTANCIA_INYECCION * (masa_inyectada / 30) ** (5 / 3)
        for posicion in POSICIONES:
            filtrado = preparar(crudos[ifo] + chirp(tiempos, posicion, masa_inyectada, distancia, F_INYECCION))
            masa_chirp, _, _ = estimar(filtrado, bancos[ifo, F_FINAL], posicion, [0, posicion], F_FINAL)
            errores.setdefault((ifo, masa_inyectada), []).append(100 * (masa_chirp / masa_inyectada - 1))

In [ ]:
for (ifo, masa_inyectada), lista in errores.items():
    lista = np.array(lista)
    print(f"{ifo}, {masa_inyectada} M☉: sesgo {lista.mean():+.1f} %, dispersión {lista.std(ddof=1):.1f} %, peor {lista[np.argmax(np.abs(lista))]:+.1f} %, dentro del 5 %: {(np.abs(lista) < 5).sum()}/{len(lista)}")
todos = np.concatenate(list(errores.values()))
print(f"Todas: sesgo {todos.mean():+.1f} %, dispersión {todos.std(ddof=1):.1f} %, dentro del 5 %: {(np.abs(todos) < 5).sum()}/{len(todos)}")

In [ ]:
for desplazamiento, ifo in [(-0.5, "H1"), (0.5, "L1")]:
    for masa_inyectada in MASAS_INYECTADAS:
        lista = errores[ifo, masa_inyectada]
        plt.plot(np.full(len(lista), masa_inyectada + desplazamiento), lista, "o", color="C0" if ifo == "H1" else "C1", label=ifo if masa_inyectada == 20 else None)
plt.axhspan(-5, 5, color="0.9")
plt.axhline(0, color="k")
plt.xlabel(r"Masa chirp inyectada (M$_\odot$)")
plt.ylabel("Error de la estimación (%)")
plt.legend();

In [ ]:
resultados = {}
for ifo in ["H1", "L1"]:
    filtrado = preparar(crudos[ifo])
    for f_final in FRECUENCIAS_FINALES:
        masa_chirp, tc, puntaje = estimar(filtrado, bancos[ifo, f_final], 0, [0], f_final)
        resultados[ifo, f_final] = masa_chirp, tc, puntaje, filtrado
        print(f"{ifo}, hasta {f_final} Hz: {masa_chirp:.2f} M☉ ({100 * (masa_chirp / MASA_CHIRP_CATALOGO - 1):+.1f} % del catálogo), fusión en t0 {tc:+.3f} s")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for eje, ifo in zip(ejes, ["H1", "L1"]):
    masa_chirp, tc, _, filtrado = resultados[ifo, F_FINAL]
    cerca = (tiempos >= -0.5) & (tiempos < 0.2)
    t_columnas, frecuencias, potencia = espectrograma(filtrado[cerca], fs, N_SEGMENTO, PASO, N_FFT)
    eje.pcolormesh(t_columnas + tiempos[cerca][0], frecuencias, potencia, shading="auto")
    traza = np.linspace(tc - 0.4, tc - 0.001, 400)
    eje.plot(traza, frecuencia_chirp(tc - traza, masa_chirp), "w--", label=rf"{masa_chirp:.2f} M$_\odot$")
    eje.set_ylim(20, 350)
    eje.set_title(ifo)
    eje.set_xlabel("Tiempo desde t0 (s)")
    eje.legend()
ejes[0].set_ylabel("Frecuencia (Hz)");

In [ ]:
for ifo in ["H1", "L1"]:
    puntaje = resultados[ifo, F_FINAL][2]
    plt.plot(MASAS, puntaje.max(axis=1) / puntaje.max(), label=ifo)
plt.axvline(MASA_CHIRP_CATALOGO, color="k", linestyle="--", label="catálogo")
plt.xlabel(r"Masa chirp (M$_\odot$)")
plt.ylabel("Puntaje relativo")
plt.legend();